In [2]:
!pip install -U transformers peft accelerate bitsandbytes datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 10.3 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.3
    Uninstalling transformers-4.57.3:
      Successfully uninstalled transformers-4.57.3
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall:

In [3]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

In [4]:
BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
ADAPTER_DIR = "/content/drive/MyDrive/coding-lora"

OUTPUT_ROOT = "/content/drive/MyDrive/Colab Notebooks/quantized"
MERGED_DIR = os.path.join(OUTPUT_ROOT, "merged-fp16")
INT8_DIR = os.path.join(OUTPUT_ROOT, "model-int8")
INT4_DIR = os.path.join(OUTPUT_ROOT, "model-int4")

os.makedirs(MERGED_DIR, exist_ok=True)
os.makedirs(INT8_DIR, exist_ok=True)
os.makedirs(INT4_DIR, exist_ok=True)

In [5]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: a6c478f0-61ef-471e-afb1-0d133b169a23)')' thrown while requesting HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [6]:
import torch
import gc

torch.cuda.empty_cache()
gc.collect()

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: cedab6a7-115b-48f2-96ea-f1716de380b3)')' thrown while requesting HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/config.json
Retrying in 1s [Retry 1/5].


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
model = PeftModel.from_pretrained(model, ADAPTER_DIR)
model = model.merge_and_unload()

model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

print("LoRA merged → FP16 saved")

LoRA merged → FP16 saved


In [9]:
bnb_int8 = BitsAndBytesConfig(load_in_8bit=True)

model_int8 = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR,
    quantization_config=bnb_int8,
    device_map="auto",
    trust_remote_code=True,
)

model_int8.save_pretrained(INT8_DIR)
tokenizer.save_pretrained(INT8_DIR)

print("INT8 saved")


INT8 saved


In [10]:
bnb_int4 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model_int4 = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR,
    quantization_config=bnb_int4,
    device_map="auto",
    trust_remote_code=True,
)

model_int4.save_pretrained(INT4_DIR)
tokenizer.save_pretrained(INT4_DIR)

print("INT4 saved")

INT4 saved


In [11]:
!git clone https://github.com/ggerganov/llama.cpp.git
print("Cloned llama.cpp")

Cloning into 'llama.cpp'...
remote: Enumerating objects: 76198, done.
remote: Counting objects: 100% (254/254), done.
remote: Compressing objects: 100% (214/214), done.
remote: Total 76198 (delta 131), reused 40 (delta 40), pack-reused 75944 (from 3)
Receiving objects: 100% (76198/76198), 279.62 MiB | 19.15 MiB/s, done.
Resolving deltas: 100% (55267/55267), done.
Cloned llama.cpp


In [12]:
!cmake -B llama.cpp/build llama.cpp
!cmake --build llama.cpp/build --config Release -j 8
print("Built llama.cpp")

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- Found OpenMP_C: 

In [13]:
!pip install -q -r llama.cpp/requirements.txt
print("Installed llma.cpp requirements")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 50.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 60.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.2/96.2 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.6/178.6 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.6/343.6 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.6/389.6 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 1.9 MB/s eta 0:00:00
   ━━━

In [14]:
!python llama.cpp/convert_hf_to_gguf.py "/content/drive/MyDrive/Colab Notebooks/quantized/merged-fp16" \
    --outfile "/content/drive/MyDrive/Colab Notebooks/quantized/model-f16.gguf" \
    --outtype f16

print("Created model-f16.gguf")

INFO:hf-to-gguf:Loading model: merged-fp16
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.float16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> F16, shape = {5632, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.float16 --> F16, shape = {2048, 256}
INFO:hf-to-gguf:blk.0.attn_output.wei

In [15]:
!./llama.cpp/build/bin/llama-quantize \
    "/content/drive/MyDrive/Colab Notebooks/quantized/model-f16.gguf" \
    "/content/drive/MyDrive/Colab Notebooks/quantized/model-q4_0.gguf" \
    Q4_0

print("Created model-q4_0.gguf")

main: build = 7759 (6ba6a3c76)
main: built with GNU 11.4.0 for Linux x86_64
main: quantizing '/content/drive/MyDrive/Colab Notebooks/quantized/model-f16.gguf' to '/content/drive/MyDrive/Colab Notebooks/quantized/model-q4_0.gguf' as Q4_0
llama_model_loader: direct I/O is enabled, disabling mmap
llama_model_loader: loaded meta data with 32 key-value pairs and 201 tensors from /content/drive/MyDrive/Colab Notebooks/quantized/model-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Merged Fp16
llama_model_loader: - kv   3:                         general.size_label str              = 1.1B
llama_model_loader: - kv   4:        

In [16]:
!./llama.cpp/build/bin/llama-quantize \
    "/content/drive/MyDrive/Colab Notebooks/quantized/model-f16.gguf" \
    "/content/drive/MyDrive/Colab Notebooks/quantized/model-q8_0.gguf" \
    Q8_0

print("Created model-q8_0.gguf")

main: build = 7759 (6ba6a3c76)
main: built with GNU 11.4.0 for Linux x86_64
main: quantizing '/content/drive/MyDrive/Colab Notebooks/quantized/model-f16.gguf' to '/content/drive/MyDrive/Colab Notebooks/quantized/model-q8_0.gguf' as Q8_0
llama_model_loader: direct I/O is enabled, disabling mmap
llama_model_loader: loaded meta data with 32 key-value pairs and 201 tensors from /content/drive/MyDrive/Colab Notebooks/quantized/model-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Merged Fp16
llama_model_loader: - kv   3:                         general.size_label str              = 1.1B
llama_model_loader: - kv   4:        

In [17]:
# Cell: Measure Model Sizes
import os

def get_size(path):
    total = 0
    if os.path.isfile(path):
        total = os.path.getsize(path)
    else:
        for root, _, files in os.walk(path):
            for f in files:
                total += os.path.getsize(os.path.join(root, f))

    return round(total / 1024 / 1024, 2)

print("=== MODEL SIZES ===")
print(f"FP16:     {get_size(MERGED_DIR)} MB")
print(f"INT8:     {get_size(INT8_DIR)} MB")
print(f"INT4:     {get_size(INT4_DIR)} MB")
print(f"GGUF-F16: {get_size(f'{OUTPUT_ROOT}/model-f16.gguf')} MB")
print(f"GGUF-Q8:  {get_size(f'{OUTPUT_ROOT}/model-q8_0.gguf')} MB")
print(f"GGUF-Q4:  {get_size(f'{OUTPUT_ROOT}/model-q4_0.gguf')} MB")


=== MODEL SIZES ===
FP16:     2102.13 MB
INT8:     1179.66 MB
INT4:     731.06 MB
GGUF-F16: 2099.05 MB
GGUF-Q8:  1115.62 MB
GGUF-Q4:  607.23 MB


In [18]:
# Cell 1: Install llama-cpp-python
!pip install -q llama-cpp-python


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 19.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.3 MB/s eta 0:00:00


In [23]:
# Cell: Measure GGUF Speed
from llama_cpp import Llama
from time import time

def test_speed(model_path, name):
    model = Llama(model_path=model_path, n_ctx=512)
    prompt = """### Instruction:
What is a stack?

### Input:
Data structure

### Output:
"""
    output = model(prompt, max_tokens=10)

    start = time()
    output = model(prompt, max_tokens=50)
    duration = time() - start

    tokens_generated = output['usage']['completion_tokens']
    tokens_per_sec = round(tokens_generated / duration, 2)

    print(f"{name}: {tokens_per_sec} tokens/sec ({duration:.2f}s, {tokens_generated} tokens)")

print("=== GGUF SPEED ===")
test_speed(f'{OUTPUT_ROOT}/model-f16.gguf', "GGUF-F16")
test_speed(f'{OUTPUT_ROOT}/model-q8_0.gguf', "GGUF-Q8")
test_speed(f'{OUTPUT_ROOT}/model-q4_0.gguf', "GGUF-Q4")

print("\n Done")


llama_model_loader: loaded meta data with 32 key-value pairs and 201 tensors from /content/drive/MyDrive/Colab Notebooks/quantized/model-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Merged Fp16
llama_model_loader: - kv   3:                         general.size_label str              = 1.1B
llama_model_loader: - kv   4:                          llama.block_count u32              = 22
llama_model_loader: - kv   5:                       llama.context_length u32              = 2048
llama_model_loader: - kv   6:                     llama.embedding_length u32              = 2048
llama_model_loader: - kv   7:              

=== GGUF SPEED ===


llama_model_loader: - kv   9:              llama.attention.head_count_kv u32              = 4
llama_model_loader: - kv  10:                       llama.rope.freq_base f32              = 10000.000000
llama_model_loader: - kv  11:     llama.attention.layer_norm_rms_epsilon f32              = 0.000010
llama_model_loader: - kv  12:                 llama.attention.key_length u32              = 64
llama_model_loader: - kv  13:               llama.attention.value_length u32              = 64
llama_model_loader: - kv  14:                          general.file_type u32              = 1
llama_model_loader: - kv  15:                           llama.vocab_size u32              = 32000
llama_model_loader: - kv  16:                 llama.rope.dimension_count u32              = 64
llama_model_loader: - kv  17:               general.quantization_version u32              = 2
llama_model_loader: - kv  18:                       tokenizer.ggml.model str              = llama
llama_model_loader: - kv  19:  

GGUF-F16: 2.9 tokens/sec (4.14s, 12 tokens)


llama_model_loader: loaded meta data with 32 key-value pairs and 201 tensors from /content/drive/MyDrive/Colab Notebooks/quantized/model-q8_0.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Merged Fp16
llama_model_loader: - kv   3:                         general.size_label str              = 1.1B
llama_model_loader: - kv   4:                          llama.block_count u32              = 22
llama_model_loader: - kv   5:                       llama.context_length u32              = 2048
llama_model_loader: - kv   6:                     llama.embedding_length u32              = 2048
llama_model_loader: - kv   7:             

GGUF-Q8: 4.34 tokens/sec (2.99s, 13 tokens)


llama_model_loader: - kv  18:                      tokenizer.ggml.tokens arr[str,32000]   = ["<unk>", "<s>", "</s>", "<0x00>", "<...
llama_model_loader: - kv  19:                      tokenizer.ggml.scores arr[f32,32000]   = [-1000.000000, -1000.000000, -1000.00...
llama_model_loader: - kv  20:                  tokenizer.ggml.token_type arr[i32,32000]   = [3, 3, 3, 6, 6, 6, 6, 6, 6, 6, 6, 6, ...
llama_model_loader: - kv  21:                tokenizer.ggml.bos_token_id u32              = 1
llama_model_loader: - kv  22:                tokenizer.ggml.eos_token_id u32              = 2
llama_model_loader: - kv  23:            tokenizer.ggml.unknown_token_id u32              = 0
llama_model_loader: - kv  24:            tokenizer.ggml.padding_token_id u32              = 2
llama_model_loader: - kv  25:               tokenizer.ggml.add_bos_token bool             = true
llama_model_loader: - kv  26:               tokenizer.ggml.add_sep_token bool             = false
llama_model_loader: - kv  27: 

GGUF-Q4: 7.27 tokens/sec (0.69s, 5 tokens)

 Done


In [20]:
# Cell: Visual Quality Test
from llama_cpp import Llama

OUTPUT_ROOT = "/content/drive/MyDrive/Colab Notebooks/quantized"

prompt = """### Instruction:
Calculate time complexity

### Input:
Algorithm: Nested loops iterating n times each

### Output:
"""

print("=== QUALITY COMPARISON ===\n")

for name, path in [
    ("F16", f'{OUTPUT_ROOT}/model-f16.gguf'),
    ("Q8", f'{OUTPUT_ROOT}/model-q8_0.gguf'),
    ("Q4", f'{OUTPUT_ROOT}/model-q4_0.gguf')
]:
    model = Llama(model_path=path, n_ctx=512)
    output = model(prompt, max_tokens=100)

    print(f"{name}:")
    print(output['choices'][0]['text'][:200])
    print()


=== QUALITY COMPARISON ===



llama_model_loader: loaded meta data with 32 key-value pairs and 201 tensors from /content/drive/MyDrive/Colab Notebooks/quantized/model-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Merged Fp16
llama_model_loader: - kv   3:                         general.size_label str              = 1.1B
llama_model_loader: - kv   4:                          llama.block_count u32              = 22
llama_model_loader: - kv   5:                       llama.context_length u32              = 2048
llama_model_loader: - kv   6:                     llama.embedding_length u32              = 2048
llama_model_loader: - kv   7:              

F16:
O(n²) - Quadratic time complexity



llama_model_loader: loaded meta data with 32 key-value pairs and 201 tensors from /content/drive/MyDrive/Colab Notebooks/quantized/model-q8_0.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Merged Fp16
llama_model_loader: - kv   3:                         general.size_label str              = 1.1B
llama_model_loader: - kv   4:                          llama.block_count u32              = 22
llama_model_loader: - kv   5:                       llama.context_length u32              = 2048
llama_model_loader: - kv   6:                     llama.embedding_length u32              = 2048
llama_model_loader: - kv   7:             

Q8:
O(n²)



llama_model_loader: loaded meta data with 32 key-value pairs and 201 tensors from /content/drive/MyDrive/Colab Notebooks/quantized/model-q4_0.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Merged Fp16
llama_model_loader: - kv   3:                         general.size_label str              = 1.1B
llama_model_loader: - kv   4:                          llama.block_count u32              = 22
llama_model_loader: - kv   5:                       llama.context_length u32              = 2048
llama_model_loader: - kv   6:                     llama.embedding_length u32              = 2048
llama_model_loader: - kv   7:             

Q4:
O(n)



In [21]:
# Cell: Test Quality HF Models
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

OUTPUT_ROOT = "/content/drive/MyDrive/Colab Notebooks/quantized"
MERGED_DIR = f"{OUTPUT_ROOT}/merged-fp16"
INT8_DIR = f"{OUTPUT_ROOT}/model-int8"
INT4_DIR = f"{OUTPUT_ROOT}/model-int4"

tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
prompt = "Write a Python function to reverse a string"

def test_quality_hf(model, tokenizer, name):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\n{'='*60}")
    print(f"{name}")
    print(f"{'='*60}")
    print(response[:300])
    return response

print("=== HF QUALITY TEST ===")
print(f"Prompt: {prompt}\n")

# FP16
model_fp16 = AutoModelForCausalLM.from_pretrained(MERGED_DIR, torch_dtype=torch.float16, device_map="auto")
test_quality_hf(model_fp16, tokenizer, "FP16")
del model_fp16
torch.cuda.empty_cache()

# INT8
model_int8 = AutoModelForCausalLM.from_pretrained(INT8_DIR, quantization_config=BitsAndBytesConfig(load_in_8bit=True), device_map="auto")
test_quality_hf(model_int8, tokenizer, "INT8")
del model_int8
torch.cuda.empty_cache()

# INT4
model_int4 = AutoModelForCausalLM.from_pretrained(INT4_DIR, quantization_config=BitsAndBytesConfig(load_in_4bit=True), device_map="auto")
test_quality_hf(model_int4, tokenizer, "INT4")
del model_int4
torch.cuda.empty_cache()

print("\nHF Quality Test Done")


`torch_dtype` is deprecated! Use `dtype` instead!


=== HF QUALITY TEST ===
Prompt: Write a Python function to reverse a string


FP16
Write a Python function to reverse a string
Explain step by step

Input: String to reverse
Output: Reversed string

def reverse_string(s):
    return s[::-1]

result = reverse_string('apple')
# Output: 'ser'


/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:239: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)



INT8
Write a Python function to reverse a string
Explain step by step

Input: String to be reversed
Output: Reversed string

def reverse_string(s):
    return s[::-1]

result = reverse_string('apple')
# Output: 'aplere'


KeyboardInterrupt: 

In [22]:
!ls -lh "/content/drive/MyDrive/Colab Notebooks/quantized/"

total 3.8G
drwx------ 2 root root 4.0K Jan 16 12:55 merged-fp16
-rw------- 1 root root 2.1G Jan 16 13:32 model-f16.gguf
drwx------ 2 root root 4.0K Jan 16 13:02 model-int4
drwx------ 2 root root 4.0K Jan 16 12:57 model-int8
-rw------- 1 root root 608M Jan 16 13:33 model-q4_0.gguf
-rw------- 1 root root 1.1G Jan 16 13:34 model-q8_0.gguf
